In [12]:
import os, re, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

DATA = "../data"
output_dir = "metadata_figs"
os.makedirs(output_dir, exist_ok=True)

def save_plot(name):
    plt.savefig(os.path.join(output_dir, f"{name}.png"), bbox_inches='tight', dpi=150)
    plt.close()

def plot_top_words(series, title, save_name, top_n=20, stop=None):
    stop = stop or {'the', 'and', 'with', 'this', 'that', 'from', 'have', 'been',
                    'they', 'their', 'which', 'will', 'also', 'some', 'into', 'its'}
    words = Counter()
    for text in series.dropna():
        for w in re.sub(r'[^\w\s]', '', str(text).lower()).split():
            if len(w) > 3 and w not in stop:
                words[w] += 1
    wdf = pd.DataFrame(words.most_common(top_n), columns=['word', 'count'])
    plt.figure(figsize=(12, 5))
    sns.barplot(data=wdf, x='count', y='word', hue='word', legend=False)
    plt.title(title)
    plt.tight_layout()
    save_plot(save_name)

print(f"Data root: {DATA}/")
print(f"Output dir: {output_dir}/")


Data root: ../data/
Output dir: metadata_figs/


## 1. SONICS Dataset

In [13]:
# --- SONICS: duration, genres, AI algorithms ---
df_fake = pd.read_csv(f"{DATA}/sonics/fake_songs.csv", low_memory=False)
df_real = pd.read_csv(f"{DATA}/sonics/real_songs.csv", low_memory=False)
df_all = pd.concat([df_fake, df_real], ignore_index=True)

print(f"Sonics: {len(df_fake)} fake, {len(df_real)} real songs")
print(f"AI algorithms: {sorted(df_fake['algorithm'].unique())}")

# Duration distribution
sns.histplot(df_all['duration'].dropna(), kde=True, color='teal')
plt.title("Sonics: Audio Duration Distribution")
plt.xlabel("Duration (s)")
save_plot("sonics_duration_dist")

# Top 15 genres
top_genres = df_all['genre'].value_counts().head(15)
plt.figure()
sns.barplot(x=top_genres.values, y=top_genres.index, hue=top_genres.index, legend=False)
plt.title("Sonics: Top 15 Genres")
plt.tight_layout()
save_plot("sonics_top_genres")

# Duration per genre (top 10)
top10 = df_all['genre'].value_counts().head(10).index
df_top = df_all[df_all['genre'].isin(top10)]
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_top, x='genre', y='duration', hue='genre', legend=False, order=top10)
plt.title("Sonics: Duration per Genre (Top 10)")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
save_plot("sonics_duration_per_genre")

# AI algorithm distribution
algo_counts = df_fake['algorithm'].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=algo_counts.values, y=algo_counts.index, hue=algo_counts.index, legend=False)
plt.title("Sonics: AI Algorithm Distribution")
plt.tight_layout()
save_plot("sonics_ai_algorithms")

print("SONICS general analysis done.")


Sonics: 49074 fake, 48090 real songs
AI algorithms: ['chirp-v2-xxl-alpha', 'chirp-v3', 'chirp-v3.5', 'udio-120s', 'udio-30s']
SONICS general analysis done.


In [14]:
# --- SONICS Lyric Pairs ---
df_map   = pd.read_csv(f"{DATA}/sonics/lyric_pairs_mapping.csv", low_memory=False)
df_pfake = pd.read_csv(f"{DATA}/sonics/pair_fake.csv", low_memory=False)
df_preal = pd.read_csv(f"{DATA}/sonics/pair_real.csv", low_memory=False)

print(f"Lyric pairs: {len(df_map)} mappings, {len(df_pfake)} fake, {len(df_preal)} real")

# Merge for duration and topic analysis
lp = (df_map
      .merge(df_pfake, left_on='fake_filename', right_on='filename')
      .merge(df_preal, left_on='real_youtube_id', right_on='youtube_id', suffixes=('_fake', '_real')))

print(f"Merged lyric pairs: {len(lp)}")

# Duration parity
plt.figure(figsize=(8, 8))
sns.scatterplot(data=lp, x='duration_real', y='duration_fake', hue='algorithm', alpha=0.5, s=15)
max_dur = max(lp['duration_real'].max(), lp['duration_fake'].max())
plt.plot([0, max_dur], [0, max_dur], 'r--', label='1:1 parity')
plt.title("Sonics Lyric Pairs: Real vs Fake Duration")
plt.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
save_plot("sonics_lyric_duration_parity")

# Lyric themes (topic)
plot_top_words(lp['topic'], "Top Lyric Topics in Sonics Lyric Pairs", "sonics_lyric_themes")

# Lyric similarity by algorithm
order = lp['algorithm'].value_counts().index.tolist()
plt.figure(figsize=(12, 5))
sns.boxplot(data=lp, x='algorithm', y='similarity', hue='algorithm', legend=False, order=order)
plt.title("Sonics: Lyric Similarity Score by Algorithm")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
save_plot("sonics_lyric_similarity_by_algo")

print("SONICS lyric pairs done.")


Lyric pairs: 462 mappings, 462 fake, 462 real
Merged lyric pairs: 462
SONICS lyric pairs done.


In [15]:
# --- SONICS Sibling Pairs ---
df_sib_map  = pd.read_csv(f"{DATA}/sonics/sibling_pairs_mapping.csv", low_memory=False)
df_sib_fake = pd.read_csv(f"{DATA}/sonics/sibling_fake.csv", low_memory=False)

print(f"Sibling pairs: {len(df_sib_map)} mappings, {len(df_sib_fake)} unique AI songs")

# Normalise filename: strip .wav if present in mapping
def norm(s):
    return str(s).replace('.wav', '')

df_sib_map['s0'] = df_sib_map['sibling_0'].apply(norm)
df_sib_map['s1'] = df_sib_map['sibling_1'].apply(norm)
df_sib_fake['filename_norm'] = df_sib_fake['filename'].apply(norm)

sib = (df_sib_map
       .merge(df_sib_fake, left_on='s0', right_on='filename_norm')
       .merge(df_sib_fake, left_on='s1', right_on='filename_norm', suffixes=('_sib0', '_sib1')))

print(f"Merged sibling pairs: {len(sib)}")

# Algorithm heatmap
algo_ct = pd.crosstab(sib['algorithm_sib0'], sib['algorithm_sib1'])
plt.figure(figsize=(max(8, len(algo_ct.columns) + 2), max(6, len(algo_ct) + 1)))
sns.heatmap(algo_ct, annot=True, fmt='d', cmap='Blues')
plt.title("Sonics Sibling Pairs: Algorithm Consistency")
plt.tight_layout()
save_plot("sonics_sibling_algo_heatmap")

# Mood heatmap (top 8)
top_moods = (pd.concat([sib['mood_sib0'], sib['mood_sib1']])
             .value_counts().head(8).index.tolist())
sib_mood = sib[sib['mood_sib0'].isin(top_moods) & sib['mood_sib1'].isin(top_moods)]
if len(sib_mood) > 0:
    mood_ct = pd.crosstab(sib_mood['mood_sib0'], sib_mood['mood_sib1'])
    plt.figure(figsize=(10, 8))
    sns.heatmap(mood_ct, annot=True, fmt='d', cmap='YlOrRd')
    plt.title("Sonics Sibling Pairs: Mood Consistency (Top 8)")
    plt.tight_layout()
    save_plot("sonics_sibling_mood_heatmap")

# Topic heatmap (top 10)
top_topics = (pd.concat([sib['topic_sib0'], sib['topic_sib1']])
              .value_counts().head(10).index.tolist())
sib_topic = sib[sib['topic_sib0'].isin(top_topics) & sib['topic_sib1'].isin(top_topics)]
if len(sib_topic) > 0:
    topic_ct = pd.crosstab(sib_topic['topic_sib0'], sib_topic['topic_sib1'])
    plt.figure(figsize=(12, 10))
    sns.heatmap(topic_ct, annot=True, fmt='d', cmap='YlGnBu')
    plt.title("Sonics Sibling Pairs: Topic Consistency (Top 10)")
    plt.tight_layout()
    save_plot("sonics_sibling_topic_heatmap")

print("SONICS sibling pairs done.")


Sibling pairs: 1000 mappings, 2000 unique AI songs
Merged sibling pairs: 1000
SONICS sibling pairs done.


## 2. FakeMusicCaps Dataset

In [16]:
# --- FakeMusicCaps ---
df_fmc = pd.read_csv(f"{DATA}/fakemusiccaps/metadata.csv")
print(f"FakeMusicCaps: {len(df_fmc)} tracks, {df_fmc['folder'].nunique()} generators")
print(df_fmc['folder'].value_counts().to_string())

# Generator distribution
plt.figure(figsize=(10, 5))
gen_counts = df_fmc['folder'].value_counts()
sns.barplot(x=gen_counts.values, y=gen_counts.index, hue=gen_counts.index, legend=False)
plt.title("FakeMusicCaps: Tracks per Generator")
plt.xlabel("Number of tracks")
plt.tight_layout()
save_plot("fmc_generator_distribution")

# Top 20 words in captions
plot_top_words(df_fmc['caption'], "Top 20 Words in FakeMusicCaps Captions",
               "top_words_in_fakemusiccaps_captions")

# Same-prompt pair coverage (same YouTube ID across generators)
def extract_yt_id(row):
    stem = Path(row['filename']).stem
    prefix = str(row['folder']) + '_'
    return stem[len(prefix):] if stem.startswith(prefix) else stem

df_fmc['youtube_id'] = df_fmc.apply(extract_yt_id, axis=1)
gen_per_yt = df_fmc.groupby('youtube_id')['folder'].nunique()
multi = df_fmc[df_fmc['youtube_id'].isin(gen_per_yt[gen_per_yt > 1].index)]
n_pairs = multi.groupby('youtube_id').apply(lambda g: len(g) * (len(g) - 1) // 2).sum()

print(f"\nUnique YouTube IDs: {df_fmc['youtube_id'].nunique()}")
print(f"YouTube IDs covered by >1 generator: {gen_per_yt[gen_per_yt > 1].shape[0]}")
print(f"Total same-prompt pairs: {n_pairs}")

# Coverage plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
vc = gen_per_yt.value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color='steelblue', edgecolor='white')
for i, (x, y) in enumerate(zip(vc.index, vc.values)):
    axes[0].text(i, y + 0.5, str(y), ha='center', va='bottom', fontsize=9)
axes[0].set_title('Generators per YouTube ID')
axes[0].set_xlabel('Number of generators covering the same prompt')
axes[0].set_ylabel('Count of YouTube IDs')

gen_in_pairs = multi.groupby('folder').size()
gen_in_pairs.sort_values().plot(kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Tracks in Same-Prompt Pairs by Generator')
axes[1].set_xlabel('Number of tracks')
plt.suptitle('FakeMusicCaps: Same-Prompt Coverage', fontsize=13)
plt.tight_layout()
save_plot("fmc_same_prompt_coverage")

print("FakeMusicCaps analysis done.")


FakeMusicCaps: 2000 tracks, 5 generators
folder
MusicGen_medium      400
audioldm2            400
musicldm             400
mustango             400
stable_audio_open    400

Unique YouTube IDs: 400
YouTube IDs covered by >1 generator: 400
Total same-prompt pairs: 4000
FakeMusicCaps analysis done.


## 3. SMP Dataset (Plagiarism Pairs)

In [17]:
# --- SMP Dataset ---
smp_csv = f"{DATA}/smp_dataset_16/Final_dataset_pairs.csv"
df_smp = pd.read_csv(smp_csv, low_memory=False)
print(f"SMP: {len(df_smp)} pairs")
print(df_smp.columns.tolist())
print(df_smp['relation'].value_counts())

# Relation type distribution
plt.figure(figsize=(10, 5))
rel_c = df_smp['relation'].value_counts()
sns.barplot(x=rel_c.values, y=rel_c.index, hue=rel_c.index, legend=False)
plt.title("SMP: Relation Types between Original and Comparison Track")
plt.xlabel("Number of pairs")
plt.tight_layout()
save_plot("smp_relation_dist")

# Duration parity from audio (if available)
import wave, unicodedata

def clean_text(text):
    if pd.isna(text): return ""
    text = unicodedata.normalize('NFKD', str(text))
    return re.sub(r'[^\w\s]', '', text).lower().strip()

def get_duration(path):
    try:
        with wave.open(str(path), 'r') as f:
            return f.getnframes() / float(f.getframerate())
    except Exception:
        return None

audio_dir = Path(f"{DATA}/smp_dataset_16")
results = []
for _, row in df_smp.iterrows():
    pair_dir = audio_dir / str(int(row['pair_number']))
    durs = {'ori': None, 'comp': None}
    if pair_dir.exists():
        target_ori  = clean_text(row['ori_title'])
        target_comp = clean_text(row['comp_title'])
        for fpath in pair_dir.glob('*.wav'):
            fname = clean_text(fpath.stem)
            if target_ori and (target_ori in fname or fname in target_ori):
                durs['ori'] = get_duration(fpath)
            elif target_comp and (target_comp in fname or fname in target_comp):
                durs['comp'] = get_duration(fpath)
    results.append(durs)

df_res = pd.concat([df_smp, pd.DataFrame(results)], axis=1).dropna(subset=['ori', 'comp'])
if len(df_res) > 0:
    plt.figure(figsize=(8, 8))
    sns.scatterplot(data=df_res, x='ori', y='comp', hue='relation', style='relation', s=60, alpha=0.7)
    max_val = max(df_res['ori'].max(), df_res['comp'].max())
    plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='1:1 parity')
    plt.title("SMP: Original vs Comparison Audio Duration")
    plt.xlabel("Original Duration (s)")
    plt.ylabel("Comparison Duration (s)")
    plt.legend()
    plt.tight_layout()
    save_plot("smp_audio_parity")
    print(f"Duration parity: {len(df_res)} pairs with audio found.")
else:
    print("Audio files not found locally — skipping duration parity plot.")

print("SMP analysis done.")


SMP: 158 pairs
['ori_title', 'comp_title', 'ori_link', 'comp_link', 'relation', 'ori_times', 'comp_times', 'pair_number', 'is_ai']
relation
remake        71
plag          66
plag_doubt    21
Name: count, dtype: int64
Duration parity: 34 pairs with audio found.
SMP analysis done.


## 4. Echoes Dataset

In [18]:
# --- Echoes ---
df_echoes = pd.read_csv(f"{DATA}/echoes_processed/processed_dataset_manifest.csv", low_memory=False)
print(f"Echoes generated tracks: {len(df_echoes)}")
print(f"Generators: {sorted(df_echoes['generator'].unique())}")
print(f"Types: {df_echoes['type'].value_counts().to_dict()}")

# Overview: generator, type, genre
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

gen_c = df_echoes['generator'].value_counts()
axes[0].barh(gen_c.index[::-1], gen_c.values[::-1], color='steelblue', edgecolor='white')
axes[0].set_title('Tracks per Generator')
axes[0].set_xlabel('Count')

type_c = df_echoes['type'].value_counts()
axes[1].bar(type_c.index, type_c.values, color=['coral', 'teal'][:len(type_c)], edgecolor='white')
axes[1].set_title('Type Distribution (ATA vs TTA)')
axes[1].set_ylabel('Count')

genre_c = df_echoes['genre'].value_counts().head(10)
axes[2].barh(genre_c.index[::-1], genre_c.values[::-1], color='mediumpurple', edgecolor='white')
axes[2].set_title('Top 10 Genres')
axes[2].set_xlabel('Count')

plt.suptitle('Echoes Dataset Overview', fontsize=13)
plt.tight_layout()
save_plot("echoes_overview")

# Duration distribution
plt.figure()
sns.histplot(df_echoes['duration'].dropna(), kde=True, color='teal', bins=40)
plt.title("Echoes: Duration Distribution")
plt.xlabel("Duration (s)")
plt.tight_layout()
save_plot("echoes_duration_dist")

# Generator vs Type heatmap
gen_type = pd.crosstab(df_echoes['generator'], df_echoes['type'])
plt.figure(figsize=(max(6, len(gen_type.columns) + 2), max(6, len(gen_type) + 1)))
sns.heatmap(gen_type, annot=True, fmt='d', cmap='Blues')
plt.title("Echoes: Generator vs Type")
plt.tight_layout()
save_plot("echoes_generator_vs_type")

# Duration by generator
order = df_echoes['generator'].value_counts().index.tolist()
plt.figure(figsize=(16, 6))
sns.boxplot(data=df_echoes, x='generator', y='duration', hue='generator', legend=False, order=order)
plt.xticks(rotation=30, ha='right')
plt.title("Echoes: Duration by Generator")
plt.tight_layout()
save_plot("echoes_generator_durations")

print("Echoes analysis done.")


Echoes generated tracks: 3577
Generators: ['acestep', 'audioldm', 'brev', 'diffrhythm', 'mubert', 'producer', 'songgen', 'stableaudio', 'suno', 'udio']
Types: {'TTA': 2569, 'ATA': 1008}
Echoes analysis done.
